# Part 4: Filtering + NUTS — pseudomarginal inference

We use the **marginal log-likelihood** from a filter as the likelihood inside MCMC: propose $\theta$, run the filter to get $\log p(y_{1:T} \mid \theta)$, and use that in the Metropolis–Hastings ratio. This is **pseudomarginal MCMC** (or particle MCMC when the filter is a particle filter): states are marginalized out, and we sample only $\theta$.

## 4.1 When is filtering + NUTS "overkill"?

When the state dimension is small and the transition/observation are simple, **sampling the full state** with NUTS + simulator (as in [Part 2](../02_dynestyx_discrete_intro.ipynb)) can be feasible and exact. Filtering then adds approximation (e.g. Taylor KF) or cost (particle filter) without much benefit.

See: [When the simulator is enough — link TBD]

## 4.2 When is filtering + NUTS valuable?

When the trajectory is **long** or the state is **high-dimensional**, sampling states is expensive or mixing is poor. Then **filtering + NUTS** (or SVI with a filter) gives parameter inference without ever sampling states.

See: [When to use filtering for parameter inference — link TBD]

## 4.3 Example: NUTS with filter (parameter-only inference)

We use the same stochastic volatility model and data as in Parts 2–3, but we **replace** `DiscreteTimeSimulator` with `FilterBasedMarginalLogLikelihood`. NUTS then samples only $\phi$; the log-likelihood at each step is the MLL from the particle filter.

In [1]:
import jax.numpy as jnp
import jax.random as jr
import numpyro
import numpyro.distributions as dist
import dynestyx as dsx
from dynestyx import Condition, Context, DynamicalModel, DiscreteTimeSimulator, Trajectory, FilterBasedMarginalLogLikelihood
from numpyro.infer import MCMC, NUTS, Predictive

def stochastic_volatility_model():
    phi = numpyro.sample("phi", dist.Uniform(0.0, 1.0))
    sigma_eta = 0.5
    initial_condition = dist.Normal(0.0, 1.0)
    def state_evolution(x, u, t_now, t_next):
        return dist.Normal(phi * x, sigma_eta)
    def observation_model(x, u, t):
        return dist.Normal(0.0, jnp.exp(x / 2.0))
    dynamics = DynamicalModel(
        state_dim=1, observation_dim=1, control_dim=0,
        initial_condition=initial_condition,
        state_evolution=state_evolution,
        observation_model=observation_model,
    )
    return dsx.sample("f", dynamics)

key = jr.PRNGKey(0)
phi_true = 0.9
obs_times = jnp.arange(0.0, 100.0, 1.0)
context_gen = Context(observations=Trajectory(times=obs_times), controls=Trajectory())
predictive = Predictive(stochastic_volatility_model, params={"phi": jnp.array(phi_true)}, num_samples=1, exclude_deterministic=False)
with DiscreteTimeSimulator():
    with Condition(context_gen):
        synthetic = predictive(key)
obs_values = synthetic["observations"][0]
observation_trajectory = Trajectory(times=obs_times, values=obs_values)

def filter_conditioned_model():
    context = Context(observations=observation_trajectory, controls=Trajectory())
    with FilterBasedMarginalLogLikelihood(filter_type="pf", n_filter_particles=1500):
        with Condition(context):
            return stochastic_volatility_model()

nuts_kernel = NUTS(filter_conditioned_model)
mcmc = MCMC(nuts_kernel, num_warmup=150, num_samples=150)
mcmc.run(jr.PRNGKey(1))
posterior_filter = mcmc.get_samples()
print("Posterior phi mean (filter+NUTS):", float(jnp.mean(posterior_filter["phi"])))
print("True phi:", phi_true)

sample: 100%|██████████| 300/300 [4:18:36<00:00, 51.72s/it, 1023 steps of size 1.61e-03. acc. prob=0.82]   

Posterior phi mean (filter+NUTS): 0.827070951461792
True phi: 0.9


**Next:** [Part 5 — SVI and warming up NUTS](../05_svi.ipynb)